# Notebook 1: AI Fundamentals, Search Techniques, and Games
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Explain what AI is and name its major subfields
- Implement BFS and DFS from scratch
- Understand heuristic-based search (A*)
- Build a Minimax-powered Tic-Tac-Toe AI
- Explain why Alpha-Beta pruning speeds up Minimax

## 1. What Is Artificial Intelligence?

AI is the science of building systems that perceive their environment and take actions that maximise their chance of achieving goals.

### Brief history
| Year | Milestone |
|------|-----------|
| 1950 | Turing publishes *Computing Machinery and Intelligence* |
| 1956 | Dartmouth Conference — "AI" coined |
| 1997 | Deep Blue defeats Kasparov at chess |
| 2012 | AlexNet wins ImageNet → deep-learning era |
| 2022+ | Large language models (GPT-4, Claude, Gemini) |

### Major subfields
- **Search & Planning** — finding sequences of actions
- **Machine Learning** — learning from data
- **Natural Language Processing** — understanding text/speech
- **Computer Vision** — interpreting images
- **Robotics** — acting in the physical world

In [ ]:
# A simple reflex agent for a two-room vacuum world
class VacuumAgent:
    def act(self, location, status):
        if status == 'Dirty':
            return 'Suck'
        return 'Move Right' if location == 'A' else 'Move Left'

agent = VacuumAgent()
scenarios = [('A','Dirty'),('A','Clean'),('B','Dirty'),('B','Clean')]
for loc, stat in scenarios:
    print(f'  Location={loc}, Status={stat}  →  {agent.act(loc, stat)}')

## 2. Problem Solving as Search

Many AI problems can be framed as **graph search**:

| Component | Meaning |
|-----------|---------|
| **State** | Snapshot of the world |
| **Initial state** | Where we start |
| **Goal state** | Where we want to be |
| **Actions** | Legal moves from a state |
| **Transition model** | New state after an action |
| **Path cost** | Total cost of a solution path |

In [ ]:
# A small undirected graph we will search
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E', 'G'],
    'G': ['F']
}
print('Nodes:', list(graph.keys()))
print('Edges from A:', graph['A'])

## 3. Breadth-First Search vs Depth-First Search

| Property | BFS | DFS |
|----------|-----|-----|
| Completeness | ✅ | ❌ (may loop) |
| Optimality (unweighted) | ✅ | ❌ |
| Time | O(b^d) | O(b^m) |
| Space | O(b^d) | O(bm) |

*b = branching factor, d = solution depth, m = max depth*

In [ ]:
from collections import deque

def bfs(graph, start, goal):
    queue = deque([[start]])
    visited = {start}
    while queue:
        path = queue.popleft()
        node = path[-1]
        if node == goal:
            return path
        for nbr in graph[node]:
            if nbr not in visited:
                visited.add(nbr)
                queue.append(path + [nbr])
    return None

def dfs(graph, start, goal, path=None, visited=None):
    path = path or [start]
    visited = visited or {start}
    if start == goal:
        return path
    for nbr in graph[start]:
        if nbr not in visited:
            visited.add(nbr)
            result = dfs(graph, nbr, goal, path + [nbr], visited)
            if result:
                return result
    return None

bfs_path = bfs(graph, 'A', 'G')
dfs_path = dfs(graph, 'A', 'G')
print(f'BFS: {" → ".join(bfs_path)}  (steps={len(bfs_path)-1})')
print(f'DFS: {" → ".join(dfs_path)}  (steps={len(dfs_path)-1})')

In [ ]:
import matplotlib.pyplot as plt
try:
    import networkx as nx
    G = nx.Graph(graph)
    pos = nx.spring_layout(G, seed=42)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, path, title in zip(axes, [bfs_path, dfs_path], ['BFS', 'DFS']):
        edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
        nx.draw_networkx(G, pos, ax=ax, node_color='#AED6F1', node_size=700, font_size=11)
        nx.draw_networkx_edges(G, pos, edgelist=edges, ax=ax, edge_color='crimson', width=3)
        nx.draw_networkx_nodes(G, pos, nodelist=path, ax=ax, node_color='#F1948A', node_size=700)
        ax.set_title(f'{title}: {" → ".join(path)}')
        ax.axis('off')
    plt.tight_layout(); plt.show()
except ImportError:
    print('Install networkx for the visualisation: pip install networkx')
    print(f'BFS path: {bfs_path}')
    print(f'DFS path: {dfs_path}')

## 4. Informed Search — A*

$$f(n) = g(n) + h(n)$$

- $g(n)$ — actual cost from start to $n$
- $h(n)$ — heuristic estimate from $n$ to goal
- $f(n)$ — estimated total path cost

A* is optimal as long as $h$ is **admissible** (never overestimates) and **consistent**.

In [ ]:
import heapq

def astar(wgraph, h, start, goal):
    # (f, g, node, path)
    pq = [(h.get(start,0), 0, start, [start])]
    best_g = {}
    while pq:
        f, g, node, path = heapq.heappop(pq)
        if node in best_g and best_g[node] <= g:
            continue
        best_g[node] = g
        if node == goal:
            return path, g
        for nbr, cost in wgraph.get(node, {}).items():
            ng = g + cost
            nf = ng + h.get(nbr, 0)
            heapq.heappush(pq, (nf, ng, nbr, path + [nbr]))
    return None, float('inf')

wgraph = {
    'A': {'B':1,'C':4}, 'B': {'A':1,'D':3,'E':2},
    'C': {'A':4,'F':5}, 'D': {'B':3},
    'E': {'B':2,'F':1}, 'F': {'C':5,'E':1,'G':2}, 'G': {}
}
h = {'A':7,'B':5,'C':6,'D':6,'E':3,'F':2,'G':0}

path, cost = astar(wgraph, h, 'A', 'G')
print(f'A* path : {" → ".join(path)}')
print(f'Total cost: {cost}')

## 5. Minimax for Games

In a two-player zero-sum game:
- **MAX** player (you) tries to *maximise* the score
- **MIN** player (opponent) tries to *minimise* your score

Both players are assumed to play **optimally**.

In [ ]:
# Perfect Tic-Tac-Toe AI via Minimax
# Board: list of 9 ints  (0=empty, 1=X, -1=O)

WINS = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def winner(b):
    for a,c,d in WINS:
        if b[a]==b[c]==b[d]!=0:
            return b[a]
    return 0 if 0 not in b else None

def minimax(b, maximising):
    w = winner(b)
    if w is not None:
        return w
    scores = []
    for i in range(9):
        if b[i] == 0:
            b[i] = 1 if maximising else -1
            scores.append(minimax(b, not maximising))
            b[i] = 0
    return max(scores) if maximising else min(scores)

def best_move(b):
    return max((i for i in range(9) if b[i]==0),
               key=lambda i: (b.__setitem__(i,1), minimax(b,False), b.__setitem__(i,0)) or minimax(b[:i]+[1]+b[i+1:], False))

# Cleaner best_move
def best_move(b):
    best_s, best_i = -float('inf'), -1
    for i in range(9):
        if b[i] == 0:
            b[i] = 1
            s = minimax(b, False)
            b[i] = 0
            if s > best_s:
                best_s, best_i = s, i
    return best_i

def show(b):
    sym = {1:'X',-1:'O',0:'.'}
    for r in range(3):
        print(' '.join(sym[b[r*3+c]] for c in range(3)))
    print()

board = [1,0,-1, 0,1,0, 0,0,-1]
print('Current board:')
show(board)
mv = best_move(board)
board[mv] = 1
print(f'AI (X) plays at index {mv}:')
show(board)
w = winner(board)
print('Result:', {1:'X wins!', -1:'O wins!', 0:'Draw'}.get(w,'Game on'))

## 6. Alpha-Beta Pruning

Minimax is correct but slow. Alpha-Beta pruning **skips branches that cannot affect the result**:

- **α** — best score MAX is guaranteed so far
- **β** — best score MIN is guaranteed so far
- Prune when **α ≥ β**

Same result as Minimax but typically **O(b^(d/2))** instead of **O(b^d)**.

In [ ]:
nodes_visited = 0

def minimax_ab(b, maximising, alpha=-float('inf'), beta=float('inf')):
    global nodes_visited
    nodes_visited += 1
    w = winner(b)
    if w is not None:
        return w
    for i in range(9):
        if b[i] == 0:
            b[i] = 1 if maximising else -1
            score = minimax_ab(b, not maximising, alpha, beta)
            b[i] = 0
            if maximising:
                alpha = max(alpha, score)
            else:
                beta = min(beta, score)
            if alpha >= beta:
                break  # prune remaining siblings
    return alpha if maximising else beta

# Count nodes for plain minimax
board_empty = [0]*9
nodes_visited = 0
minimax(board_empty, True)
print(f'Plain Minimax nodes evaluated  : ~{nodes_visited:,}')

nodes_visited = 0
minimax_ab(board_empty, True)
print(f'Alpha-Beta Pruning nodes evaluated: ~{nodes_visited:,}')

## Exercises

1. **Modify BFS** to also print the order in which nodes were visited. How does it differ from DFS?
2. **Heuristic design**: For the 8-puzzle (3×3 sliding tiles), define an admissible heuristic and justify why it never overestimates.
3. **Alpha-Beta depth**: If the game tree has branching factor `b=3` and depth `d=6`, how many nodes does plain Minimax visit? How many does Alpha-Beta visit in the best case?
4. **Extend the AI**: Change the Tic-Tac-Toe AI so it plays as `O` (the minimising player). Play a game against it.

## Reflection Questions

- Why is BFS guaranteed to find the shortest path but DFS is not?
- What does it mean for a heuristic to be *consistent* (monotone)?
- Is an AI that plays perfect Tic-Tac-Toe "intelligent" in any meaningful sense?

---
**Next →** `02_numpy_essentials.ipynb`